In [23]:
import pandas as pd
import numpy as np

In [24]:
mca_df = pd.read_csv('../data/auxiliary/final_mca_soc_code.csv')
mca_df["Job Sector"] = (
    mca_df["Job Sector"].replace(
        {
            "Arts, entertainment and recreation": "Arts, Entertainment and Recreation",
            "Professional, Scientific And Technical Activities" : "Professional, Scientific and Technical Activities"
        }
    )
)

ai_measurements_df = pd.read_csv('../data/auxiliary/stage_3_archetypes.csv')

In [25]:
mca_AI_df = pd.read_csv('../data/auxiliary/final_mca_with_ai_measurements.csv')

In [26]:
excel_tables = {}

# 1. Creating Occupation Master

| Column name          | Value or calculation                                                                           |
| -------------------- | ---------------------------------------------------------------------------------------------- |
| `SOC_code`           | Occupation code used in the existing analysis.                                                 |
| `SOC_title`          | Corresponding occupation title.                                                                |
| `Sector`             | Sector assigned in the existing analysis.                                                      |
| `AI_potential_score` | Existing value used for the potential axis.                                                    |
| `AI_observed_score`  | Existing value used for the observed-use axis.                                                 |
| `AI_archetype`       | Existing classification in the current run.                                                    |
| `MCA_entry_count`    | Number of distinct MCA career entries mapped to this SOC code under the existing mapping rule. |
| `In_MCA`             | Yes if `MCA_entry_count` > 0; otherwise No.                                                    |
| `Potential_imputed`  | Yes, No, or Unknown.                                                                           |
| `Observed_imputed`   | Yes, No, or Unknown.                                                                           |

Archetype C-AIOE Percentile - Observed Exposure 80

In [53]:
rename_columns = {
    'O*NET-SOC Code': 'SOC_code',
    'Title': 'SOC_title',
    'Major Group': 'Sector',
    'Archetype AIOE Percentile - Observed Exposure 50': 'Archetype',
}

occupation_master = ai_measurements_df.rename(
    columns=rename_columns
).copy()

occupation_master = occupation_master[
    list(rename_columns.values())
    + [
        'AIOE',
        'Complementarity', 
        'C-AIOE',
        'LLM Rating Beta',
        'Human Rating Beta',
        'Usage Rate',
        'Observed Exposure',
        'Augmentation Percent',
        'Automation Percent'
    ]
]

occupation_master['AIOE Percentile'] = occupation_master['AIOE'].rank(pct=True)

mca_titles_dict = (
    mca_df.groupby('SOC Code')['Job Title']
    .agg(lambda titles: '; '.join(titles.dropna().unique()))
    .to_dict()
)
occupation_master['MCA_titles'] = (
    occupation_master['SOC_code']
    .map(mca_titles_dict)
    .fillna('None')
)
occupation_master['MCA_entry_count'] = (
    occupation_master['SOC_code']
    .apply(lambda x: len([
        title for title in mca_titles_dict.get(x, '').split(';')
        if title.strip()
    ]))
)
occupation_master['In_MCA'] = occupation_master.apply(lambda row : 'No' if row['MCA_entry_count'] == 0 else 'Yes', axis=1)

In [54]:
mca_titles_dict.get('11-1021.00', ['']).split(';')

['Newsroom Assistant', ' Store Keeper']

In [55]:
mca_titles_dict.get('ff', '').split(';')

['']

In [56]:
excel_tables['Occupation_Master'] = occupation_master

# 2. Archetype Comparison

| Column name                | Value or calculation                                                            |
| -------------------------- | ------------------------------------------------------------------------------- |
| `AI_archetype`             | Existing archetype name.                                                        |
| `SOC_count`                | Number of comparison-sample SOC occupations in the archetype.                   |
| `SOC_share_pct`            | `SOC_count` ÷ total classified comparison-sample SOC occupations × 100.         |
| `MCA_entry_count`          | Number of distinct classified MCA entries in the archetype.                     |
| `MCA_entry_share_pct`      | `MCA_entry_count` ÷ total classified MCA entries × 100.                         |
| `MCA_unique_SOC_count`     | Number of distinct classified MCA-matched SOC codes in the archetype.           |
| `MCA_unique_SOC_share_pct` | `MCA_unique_SOC_count` ÷ total distinct classified MCA-matched SOC codes × 100. |
| `Entry_gap_pp`             | `MCA_entry_share_pct` − `SOC_share_pct`.                                        |
| `Unique_SOC_gap_pp`        | `MCA_unique_SOC_share_pct` − `SOC_share_pct`.                                   |

In [57]:
archetype_column = 'Archetype AIOE Percentile - Observed Exposure 50'

# SOC
soc_entry = ai_measurements_df.groupby(
    archetype_column
)['O*NET-SOC Code'].count()

soc_entry_pct = (soc_entry / soc_entry.sum()) * 100

# MCA Entry
mca_entry = mca_AI_df.groupby(
    archetype_column
)['SOC Code'].count()

mca_entry_pct = (mca_entry / mca_entry.sum()) * 100

# MCA Unique SOC
mca_unique_entry = (
    mca_AI_df
    .drop_duplicates(subset=['SOC Code'])
    .groupby(archetype_column)['SOC Code']
    .count()
)

mca_unique_entry_pct = (
    mca_unique_entry / mca_unique_entry.sum()
) * 100

# Combine metrics
archetype_comparison_df = pd.concat(
    [
        soc_entry,
        soc_entry_pct,
        mca_entry,
        mca_entry_pct,
        mca_unique_entry,
        mca_unique_entry_pct
    ],
    axis=1
).reset_index()

archetype_comparison_df.columns = [
    'AI_archetype',
    'SOC_count',
    'SOC_share_pct',
    'MCA_entry_count',
    'MCA_entry_share_pct',
    'MCA_unique_SOC_count',
    'MCA_unique_SOC_share_pct'
]

# Calculate percentage-point gaps
archetype_comparison_df['Entry_gap_pp'] = (
    archetype_comparison_df['MCA_entry_share_pct']
    - archetype_comparison_df['SOC_share_pct']
)

archetype_comparison_df['Unique_SOC_gap_pp'] = (
    archetype_comparison_df['MCA_unique_SOC_share_pct']
    - archetype_comparison_df['SOC_share_pct']
)

archetype_meanings = {
    'Ahead of Estimate': 'Low AIOE, High Observed Exposure',
    'In Motion': 'High AIOE, High Observed Exposure',
    'Protected for Now': 'Low AIOE, Low Observed Exposure',
    'Untapped': 'High AIOE, Low Observed Exposure'
}

archetype_comparison_df.insert(
    1,
    'AI_archetype_meaning',
    archetype_comparison_df['AI_archetype'].map(archetype_meanings)
)

# add an order to the archetypes
archetype_order = [
    'Protected for Now',
    'Ahead of Estimate',
    'Untapped',
    'In Motion'
]

archetype_comparison_df['AI_archetype'] = pd.Categorical(
    archetype_comparison_df['AI_archetype'],
    categories=archetype_order,
    ordered=True
)

archetype_comparison_df = (
    archetype_comparison_df
    .sort_values('AI_archetype')
    .reset_index(drop=True)
)

In [58]:
excel_tables['Archetype_Comparison'] = archetype_comparison_df

# 3. Sector Comparison

| Column                            | Definition                                                                |
| --------------------------------- | ------------------------------------------------------------------------- |
| `Sector`                          | Existing sector name. Use `Unknown` for unassigned sectors and flag them. |
| `SOC_total`                       | All classified comparison-sample occupations in this sector.              |
| `SOC_active_count`                | Comparison-sample occupations classified as Active Transformation.        |
| `SOC_active_pct`                  | `SOC_active_count` ÷ `SOC_total` × 100.                                   |
| `MCA_entries_total`               | All distinct classified MCA entries in this sector.                       |
| `MCA_entries_active_count`        | MCA entries classified as Active Transformation.                          |
| `MCA_entries_active_pct`          | `MCA_entries_active_count` ÷ `MCA_entries_total` × 100.                   |
| `MCA_unique_SOC_total`            | Distinct classified MCA-matched SOC codes in this sector.                 |
| `MCA_unique_SOC_active_count`     | Distinct MCA-matched SOC codes classified as Active Transformation.       |
| `MCA_unique_SOC_active_pct`       | `MCA_unique_SOC_active_count` ÷ `MCA_unique_SOC_total` × 100.             |
| `Within_sector_gap_pp`            | `MCA_unique_SOC_active_pct` − `SOC_active_pct`.                           |
| `Share_of_MCA_active_entries_pct` | `MCA_entries_active_count` ÷ all MCA Active Transformation entries × 100. |

 - Observed Exposure

In [59]:
ai_archetype_column = 'Archetype AIOE Percentile - Observed Exposure 50'

archetypes = archetype_order

# SOC comparison sample
soc_summary = (
    ai_measurements_df
    .groupby('Major Group')
    .agg(SOC_total=('O*NET-SOC Code', 'count'))
)

soc_archetype_counts = pd.crosstab(
    ai_measurements_df['Major Group'],
    ai_measurements_df[ai_archetype_column]
).reindex(columns=archetypes, fill_value=0)

soc_archetype_counts.columns = [
    f'SOC_{archetype.replace(" ", "_")}_count'
    for archetype in soc_archetype_counts.columns
]

soc_summary = soc_summary.join(soc_archetype_counts)

for archetype in archetypes:
    column = f'SOC_{archetype.replace(" ", "_")}_count'
    soc_summary[column.replace('_count', '_pct')] = (
        soc_summary[column] / soc_summary['SOC_total'] * 100
    )

# MCA entries
mca_summary = (
    mca_AI_df
    .groupby('Major Group')
    .agg(MCA_entries_total=('SOC Code', 'count'))
)

mca_archetype_counts = pd.crosstab(
    mca_AI_df['Major Group'],
    mca_AI_df[ai_archetype_column]
).reindex(columns=archetypes, fill_value=0)

mca_archetype_counts.columns = [
    f'MCA_entries_{archetype.replace(" ", "_")}_count'
    for archetype in mca_archetype_counts.columns
]

mca_summary = mca_summary.join(mca_archetype_counts)

for archetype in archetypes:
    column = f'MCA_entries_{archetype.replace(" ", "_")}_count'
    mca_summary[column.replace('_count', '_pct')] = (
        mca_summary[column] / mca_summary['MCA_entries_total'] * 100
    )

# MCA unique SOC codes
mca_unique_df = mca_AI_df.drop_duplicates(
    subset=['SOC Code']
)

mca_unique_summary = (
    mca_unique_df
    .groupby('Major Group')
    .agg(MCA_unique_SOC_total=('SOC Code', 'count'))
)

mca_unique_archetype_counts = pd.crosstab(
    mca_unique_df['Major Group'],
    mca_unique_df[ai_archetype_column]
).reindex(columns=archetypes, fill_value=0)

mca_unique_archetype_counts.columns = [
    f'MCA_unique_SOC_{archetype.replace(" ", "_")}_count'
    for archetype in mca_unique_archetype_counts.columns
]

mca_unique_summary = mca_unique_summary.join(
    mca_unique_archetype_counts
)

for archetype in archetypes:
    column = f'MCA_unique_SOC_{archetype.replace(" ", "_")}_count'
    mca_unique_summary[column.replace('_count', '_pct')] = (
        mca_unique_summary[column]
        / mca_unique_summary['MCA_unique_SOC_total'] * 100
    )

# Combine summaries
sector_comparison_df = (
    soc_summary
    .join(mca_summary, how='outer')
    .join(mca_unique_summary, how='outer')
    .fillna(0)
    .reset_index()
    .rename(columns={'Major Group': 'Sector'})
)

# Order columns by dataset, keeping each count next to its percentage
soc_columns = ['SOC_total']
mca_entry_columns = ['MCA_entries_total']
mca_unique_columns = ['MCA_unique_SOC_total']

for archetype in archetypes:
    suffix = archetype.replace(' ', '_')

    soc_columns.extend([
        f'SOC_{suffix}_count',
        f'SOC_{suffix}_pct'
    ])

    mca_entry_columns.extend([
        f'MCA_entries_{suffix}_count',
        f'MCA_entries_{suffix}_pct'
    ])

    mca_unique_columns.extend([
        f'MCA_unique_SOC_{suffix}_count',
        f'MCA_unique_SOC_{suffix}_pct'
    ])

sector_comparison_df = sector_comparison_df[
    ['Sector']
    + soc_columns
    + mca_entry_columns
    + mca_unique_columns
]

sector_comparison_df = sector_comparison_df.sort_values(
    'Sector'
).reset_index(drop=True)

sector_comparison_df

,Sector,SOC_total,SOC_Protected_for_Now_count,SOC_Protected_for_Now_pct,SOC_Ahead_of_Estimate_count,SOC_Ahead_of_Estimate_pct,SOC_Untapped_count,SOC_Untapped_pct,SOC_In_Motion_count,SOC_In_Motion_pct,...,MCA_entries_In_Motion_pct,MCA_unique_SOC_total,MCA_unique_SOC_Protected_for_Now_count,MCA_unique_SOC_Protected_for_Now_pct,MCA_unique_SOC_Ahead_of_Estimate_count,MCA_unique_SOC_Ahead_of_Estimate_pct,MCA_unique_SOC_Untapped_count,MCA_unique_SOC_Untapped_pct,MCA_unique_SOC_In_Motion_count,MCA_unique_SOC_In_Motion_pct
0,Architecture and Engineering,31,7,22.580645,0,0.000000,22,70.967742,2,6.451613,...,1.639344,17,4,23.529412,0,0.000000,12,70.588235,1,5.882353
1,"Arts, Design, Entertainment, Sports, and Media",33,15,45.454545,4,12.121212,4,12.121212,10,30.303030,...,48.543689,26,12,46.153846,2,7.692308,3,11.538462,9,34.615385
2,Building and Grounds Cleaning and Maintenance,4,4,100.000000,0,0.000000,0,0.000000,0,0.000000,...,0.000000,2,2,100.000000,0,0.000000,0,0.000000,0,0.000000
3,Business and Financial Operations,26,1,3.846154,0,0.000000,12,46.153846,13,50.000000,...,90.322581,11,1,9.090909,0,0.000000,2,18.181818,8,72.727273
4,Community and Social Service,12,1,8.333333,0,0.000000,9,75.000000,2,16.666667,...,3.030303,6,0,0.000000,0,0.000000,5,83.333333,1,16.666667
5,Computer and Mathematical,19,0,0.000000,1,5.263158,1,5.263158,17,89.473684,...,76.923077,14,0,0.000000,1,7.142857,1,7.142857,12,85.714286
6,Construction and Extraction,14,14,100.000000,0,0.000000,0,0.000000,0,0.000000,...,0.000000,8,8,100.000000,0,0.000000,0,0.000000,0,0.000000
7,Educational Instruction and Library,56,7,12.500000,2,3.571429,25,44.642857,22,39.285714,...,50.769231,20,5,25.000000,1,5.000000,4,20.000000,10,50.000000
8,"Farming, Fishing, and Forestry",4,4,100.000000,0,0.000000,0,0.000000,0,0.000000,...,0.000000,4,4,100.000000,0,0.000000,0,0.000000,0,0.000000
9,Food Preparation and Serving Related,13,13,100.000000,0,0.000000,0,0.000000,0,0.000000,...,0.000000,6,6,100.000000,0,0.000000,0,0.000000,0,0.000000


In [60]:
excel_tables['Sector_Comparison'] = sector_comparison_df

# 4. Add tabs for each archetype

In [61]:
for archetype in archetype_order:
    is_archetype = occupation_master['Archetype'] == archetype
    archetype_df = occupation_master[is_archetype]
    excel_tables[f'{archetype} Jobs'] = archetype_df

In [62]:
archetype_df

,SOC_code,SOC_title,Sector,Archetype,AIOE,Complementarity,C-AIOE,LLM Rating Beta,Human Rating Beta,Usage Rate,Observed Exposure,Augmentation Percent,Automation Percent,AIOE Percentile,MCA_titles,MCA_entry_count,In_MCA
96,43-9031.00,Desktop Publishers,Office and Administrative Support,In Motion,6.320767,49.125000,5.608364,0.645161,0.483871,0.50,0.4640,42.71,57.29,0.512104,None,0,No
112,43-4081.00,"Hotel, Motel, and Resort Desk Clerks",Office and Administrative Support,In Motion,6.307203,54.666667,5.246804,0.500000,0.397436,0.00,0.1725,51.06,48.94,0.506518,Guest Relations Associate,1,Yes
115,43-9022.00,Word Processors and Typists,Office and Administrative Support,In Motion,6.305937,55.083333,5.219477,0.743243,0.486486,1.05,0.2385,43.10,56.90,0.504655,Court Stenographer,1,Yes
116,27-1024.00,Graphic Designers,"Arts, Design, Entertainment, Sports, and Media",In Motion,6.371819,54.000000,5.343035,0.500000,0.410714,1.08,0.3672,69.38,30.62,0.545624,Multimedia Designer; Publishing Coordinator; J...,6,Yes
118,43-9061.00,"Office Clerks, General",Office and Administrative Support,In Motion,6.479838,48.729167,5.775155,0.575758,0.500000,0.70,0.4504,29.07,70.93,0.629423,Embassy Administrative Attaché; Ministry Progr...,10,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
471,25-2022.00,"Middle School Teachers, Except Special and Car...",Educational Instruction and Library,In Motion,6.617443,70.479167,4.458502,0.347826,0.340580,0.02,0.2972,63.35,36.65,0.819367,None,0,No
476,25-1052.00,"Chemistry Teachers, Postsecondary",Educational Instruction and Library,In Motion,6.635993,70.854167,4.446115,0.431373,0.401961,0.01,0.2185,71.25,28.75,0.856611,None,0,No
492,11-1021.00,General and Operations Managers,Management,In Motion,6.500937,73.250000,4.199876,0.480769,0.384615,0.20,0.1378,56.68,43.32,0.644320,Newsroom Assistant; Store Keeper,2,Yes
495,25-1042.00,"Biological Science Teachers, Postsecondary",Educational Instruction and Library,In Motion,6.588080,73.479167,4.241076,0.469388,0.397959,0.11,0.2914,69.26,30.74,0.769088,Anatomy and Physiology Instructor; Biology Ins...,2,Yes


# 5. Checks and submission

| Check                                  | Expected                                                                                                    | Actual | Pass/Flag | Notes |
| -------------------------------------- | ----------------------------------------------------------------------------------------------------------- | ------ | --------- | ----- |
| Comparison sample count                | 537 total occupations                                                                                       |        |           |       |
| Comparison Active Transformation count | 139 occupations                                                                                             |        |           |       |
| MCA classified entry count             | 852 entries                                                                                                 |        |           |       |
| MCA Active Transformation count        | 322 entries                                                                                                 |        |           |       |
| Archetype count totals                 | Each count column sums to its declared denominator                                                          |        |           |       |
| Archetype share totals                 | Each share column sums to 100%, within a rounding tolerance of 0.1 × number of archetypes percentage points |        |           |       |
| Sector count totals                    | Each total agrees with its corresponding archetype-table count                                              |        |           |       |
| Unique master codes                    | Zero duplicate SOC codes in `Occupation_Master`                                                             |        |           |       |
| Common thresholds                      | Identical definitions, thresholds, transformations, and boundary rules across the two samples               |        |           |       |
| Six occupation rows                    | Exactly six requested occupations, with no duplicates                                                       |        |           |       |
| Expansion run counts                   | `Expansion_runs` ≤ `Available_runs`; all counted runs appear in the run key                                 |        |           |       |

In [63]:
537 == occupation_master.SOC_code.nunique()

True

In [64]:
139 == occupation_master['Archetype'].eq('In Motion').sum()

np.False_

In [65]:
852 == mca_AI_df['Archetype AIOE Percentile - Observed Exposure 50'].dropna().shape[0]

True

In [66]:
322 == mca_AI_df['Archetype AIOE Percentile - Observed Exposure 50'].eq('In Motion').sum()

np.False_

In [67]:
share_totals = (
    archetype_comparison_df
    [[
        'SOC_share_pct',
        'MCA_entry_share_pct',
        'MCA_unique_SOC_share_pct'
    ]]
    .sum()
)

checks = np.isclose(share_totals, 100, atol=0.1)

checks.all()

np.True_

In [68]:
# active_archetype_comparison_all_df = archetype_comparison_df[
#     archetype_comparison_df['AI_archetype'] == 'In Motion'] 

# active_archetype_totals = (
#     active_archetype_comparison_all_df
#     [[
#         'SOC_count',
#         'MCA_entry_count',
#         'MCA_unique_SOC_count'
#     ]]
#     .sum()
# )

# sector_totals = sector_comparison_df[[
#     'SOC_active_count',
#     'MCA_entries_active_count',
#     'MCA_unique_SOC_active_count'
# ]].sum()

# sector_archetype_totals = pd.DataFrame({
#     'Archetype_Total': active_archetype_totals.to_numpy(),
#     'Sector_Total': sector_totals.to_numpy(),
# }, index=[
#     'SOC_count',
#     'MCA_entry_count',
#     'MCA_unique_SOC_count'
# ])

# sector_archetype_totals['Is_equal'] = (
#     sector_archetype_totals['Archetype_Total']
#     == sector_archetype_totals['Sector_Total']
# )

# sector_archetype_totals['Is_equal'].all()

In [69]:
occupation_master.drop_duplicates(subset='SOC_code').shape == occupation_master.shape

True

In [70]:
# (six_occupations_comparison_df['Expansion_runs'] <= six_occupations_comparison_df['Available_runs']).all()

In [71]:
# checks_df = pd.DataFrame([
#     {
#         'Check': 'Comparison sample count',
#         'Expected': '537 total occupations',
#         'Actual': occupation_master['SOC_code'].nunique(),
#         'Pass/Flag': (
#             'Pass' if occupation_master['SOC_code'].nunique() == 537
#             else 'Flag'
#         ),
#         'Notes': ''
#     },
#     {
#         'Check': 'Comparison Active Transformation count',
#         'Expected': '139 occupations',
#         'Actual': occupation_master[
#             'Archetype C-AIOE Percentile - Observed Exposure 50'
#         ].eq('Active Transformation').sum(),
#         'Pass/Flag': (
#             'Pass' if occupation_master[
#                 'Archetype C-AIOE Percentile - Observed Exposure 50'
#             ].eq('Active Transformation').sum() == 139
#             else 'Flag'
#         ),
#         'Notes': ''
#     },
#     {
#         'Check': 'MCA classified entry count',
#         'Expected': '852 entries',
#         'Actual': mca_AI_df[
#             'Archetype C-AIOE Percentile - Observed Exposure 50'
#         ].notna().sum(),
#         'Pass/Flag': (
#             'Pass' if mca_AI_df[
#                 'Archetype C-AIOE Percentile - Observed Exposure 50'
#             ].notna().sum() == 852
#             else 'Flag'
#         ),
#         'Notes': ''
#     },
#     {
#         'Check': 'MCA Active Transformation count',
#         'Expected': '322 entries',
#         'Actual': mca_AI_df[
#             'Archetype C-AIOE Percentile - Observed Exposure 50'
#         ].eq('Active Transformation').sum(),
#         'Pass/Flag': (
#             'Pass' if mca_AI_df[
#                 'Archetype C-AIOE Percentile - Observed Exposure 50'
#             ].eq('Active Transformation').sum() == 322
#             else 'Flag'
#         ),
#         'Notes': ''
#     },
#     {
#         'Check': 'Archetype count totals',
#         'Expected': 'Each count column sums to its declared denominator',
#         'Actual': (
#             'Pass' if (
#                 archetype_comparison_all_df
#                 .groupby('Threshold')[[
#                     'SOC_count',
#                     'MCA_entry_count',
#                     'MCA_unique_SOC_count'
#                 ]].sum().eq(
#                     archetype_comparison_all_df.groupby('Threshold')[
#                         'SOC_count'
#                     ].sum().to_numpy()[:, None]
#                 ).all().all()
#             ) else 'See Notes'
#         ),
#         'Pass/Flag': 'Pass' if (
#             archetype_comparison_all_df.groupby('Threshold')[
#                 'SOC_count'
#             ].sum().eq(537).all()
#             and archetype_comparison_all_df.groupby('Threshold')[
#                 'MCA_entry_count'
#             ].sum().eq(852).all()
#         ) else 'Flag',
#         'Notes': 'Check each count column against its own denominator.'
#     },
#     {
#         'Check': 'Archetype share totals',
#         'Expected': 'Each share column sums to 100%, within tolerance',
#         'Actual': share_totals.round(4).to_dict('index'),
#         'Pass/Flag': 'Pass' if checks.all() else 'Flag',
#         'Notes': 'Tolerance: 0.1 percentage points.'
#     },
#     {
#         'Check': 'Sector count totals',
#         'Expected': 'Each total agrees with its corresponding archetype-table count',
#         'Actual': sector_archetype_totals['Is_equal'].to_dict(),
#         'Pass/Flag': (
#             'Pass' if sector_archetype_totals['Is_equal'].all()
#             else 'Flag'
#         ),
#         'Notes': ''
#     },
#     {
#         'Check': 'Unique master codes',
#         'Expected': 'Zero duplicate SOC codes in Occupation_Master',
#         'Actual': (
#             'No duplicates' if occupation_master['SOC_code'].is_unique
#             else f"{occupation_master['SOC_code'].duplicated().sum()} duplicates"
#         ),
#         'Pass/Flag': (
#             'Pass' if occupation_master['SOC_code'].is_unique
#             else 'Flag'
#         ),
#         'Notes': ''
#     },
#     {
#         'Check': 'Common thresholds',
#         'Expected': 'Identical definitions, thresholds, transformations, and boundary rules',
#         'Actual': 'Requires manual verification',
#         'Pass/Flag': 'Flag',
#         'Notes': 'Confirm shared classification logic across both samples.'
#     },
#     {
#         'Check': 'Six occupation rows',
#         'Expected': 'Exactly six requested occupations, with no duplicates',
#         'Actual': f'{len(six_occupations_comparison_df)} rows',
#         'Pass/Flag': (
#             'Pass' if (
#                 len(six_occupations_comparison_df) == 6
#                 and not six_occupations_comparison_df.duplicated().any()
#             ) else 'Flag'
#         ),
#         'Notes': ''
#     },
#     {
#         'Check': 'Expansion run counts',
#         'Expected': 'Expansion_runs <= Available_runs; all counted runs appear in the run key',
#         'Actual': (
#             'Inequality satisfied'
#             if (
#                 six_occupations_comparison_df['Expansion_runs']
#                 <= six_occupations_comparison_df['Available_runs']
#             ).all()
#             else 'Inequality violated'
#         ),
#         'Pass/Flag': (
#             'Pass' if (
#                 six_occupations_comparison_df['Expansion_runs']
#                 <= six_occupations_comparison_df['Available_runs']
#             ).all()
#             else 'Flag'
#         ),
#         'Notes': 'Run-key membership must be checked separately.'
#     }
# ])

# excel_tables['Checks'] = checks_df

# Make the excel File

In [72]:
from pathlib import Path

import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

output_path = Path('../data/AI_Research_Technical_Outputs.xlsx')
output_path = Path('AI_Research_Technical_Outputs.xlsx')

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for sheet_name, df in excel_tables.items():
        df.to_excel(
            writer,
            sheet_name=sheet_name[:31],
            index=False,
        )

        ws = writer.sheets[sheet_name[:31]]
        ws.freeze_panes = 'A2'

        # Format the header row.
        for cell in ws[1]:
            cell.font = Font(bold=True, color='FFFFFF')
            cell.fill = PatternFill(
                fill_type='solid',
                fgColor='1F4E78',
            )
            cell.alignment = Alignment(
                horizontal='left',
                vertical='center',
                wrap_text=True,
            )

        ws.row_dimensions[1].height = 32

        # Wrap cell contents and align values vertically.
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(
                    vertical='top',
                    wrap_text=True,
                )

        # Set readable column widths based on the actual contents.
        for column_idx, column_cells in enumerate(
            ws.iter_cols(),
            start=1,
        ):
            max_length = max(
                (
                    len(str(cell.value))
                    for cell in column_cells
                    if cell.value is not None
                ),
                default=0,
            )

            header = str(ws.cell(1, column_idx).value or '')
            width = min(max(max_length + 2, len(header) + 2), 45)

            ws.column_dimensions[
                get_column_letter(column_idx)
            ].width = width

        # Keep a final total row outside the filter range.
        last_row = ws.max_row
        filter_last_row = last_row

        if last_row > 1:
            first_value = str(
                ws.cell(last_row, 1).value or ''
            ).strip().lower()

            if first_value in {
                'total',
                'totals',
                'grand total',
                'overall total',
            }:
                filter_last_row -= 1

                for cell in ws[last_row]:
                    cell.font = Font(bold=True)
                    cell.fill = PatternFill(
                        fill_type='solid',
                        fgColor='D9EAF7',
                    )

        if filter_last_row >= 1 and ws.max_column > 0:
            ws.auto_filter.ref = (
                f'A1:{get_column_letter(ws.max_column)}'
                f'{filter_last_row}'
            )

print(f'Excel file saved to: {output_path.resolve()}')

Excel file saved to: /Users/jfv/Desktop/wadhwani/10_AI_research/04_creating_output/AI_Research_Technical_Outputs.xlsx
